# Non-YOLO detector row: RT-DETR-L under the fixed-epoch protocol (official split vs HBFMID-Clean)

Adds a transformer detector to the benchmark so the paper is not "YOLO only" and so we learn whether a different architecture family beats YOLO on the leakage-free protocol. Same data lists, same fixed protocol as notebook 08: 100 epochs, no early stopping, `last.pt` (EMA) evaluated, Ultralytics defaults (no per-model tuning; AdamW is auto-selected for both families at this dataset size).

* **PART = "A"**: `rtdetr-l.pt`, seed 42, official split + 5 clean folds = 6 runs.
* **PART = "B"** (only if A looks worth it): seeds 43 and 44, official + clean = 12 runs.

**How to run (Kaggle, GPU T4 x2, Internet ON, HBFMID dataset attached):**
1. Set `SMOKE = True`, run all cells (about 15 to 25 min). It trains 2 epochs on official + fold 0 and must end with "failed runs: none" and a metrics table.
2. Note the printed seconds per epoch (in `metrics.json: train_seconds`) and multiply by 100 to predict the real run time.
3. Set `SMOKE = False`, Save Version -> Save & Run All (background).
4. If a log shows `nan` losses, set `NOAMP = True` (fp32, slower) and rerun that run.

Caveats stated up front: RT-DETR is run with the default Ultralytics recipe, untuned; one seed in PART A means the detectable difference is about 1.6 points (see docs/RESULTS_BENCHMARK.md).

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert "rtdetr" in open(f"{REPO}/scripts/training/run_baseline.py").read(), "repo is behind: push the run_baseline.py RT-DETR patch first"
# download the pretrained weights ONCE here (cwd is inherited by the workers) so the two GPU workers do not race on the download
from ultralytics import RTDETR, YOLO
RTDETR("rtdetr-l.pt"); YOLO("yolo11n.pt")          # yolo11n.pt is fetched by the AMP self-check
print("weights:", [f for f in os.listdir(".") if f.endswith(".pt")])

In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = True                   # True first (2 epochs, official + fold 0), then False for the real run
PART = "A"                     # "A" or "B"
NOAMP = False                  # True = fp32 training (use if RT-DETR shows NaN losses)
BATCH = 16                     # same as the YOLO runs; if CUDA out-of-memory use 8 and say so in the paper
MODEL = "rtdetr-l.pt"
JOBS = {"A": [(MODEL, [42], [42])],
        "B": [(MODEL, [43, 44], [43, 44])]}[PART]                    # (model, official seeds, clean seeds)
FOLDS = [0, 1, 2, 3, 4]
if SMOKE:
    EPOCHS, OUT = 2, f"/kaggle/working/results/rtdetr_{PART}_smoke"
    JOBS = [(m, o[:1], c[:1]) for m, o, c in JOBS][:1]; FOLDS = [0]
else:
    EPOCHS, OUT = 100, f"/kaggle/working/results/rtdetr_{PART}"

def make_cmd(model, mode, fold, seed):
    def f(gpu):
        cmd = [sys.executable, f"{REPO}/scripts/training/run_baseline.py", "--mode", mode, "--fold", str(fold), "--seed", str(seed),
               "--model", model, "--epochs", str(EPOCHS), "--batch", str(BATCH), "--fixed-epochs", "--device", str(gpu),
               "--data-root", DATA_ROOT, "--splits-dir", f"{REPO}/data_splits", "--out", OUT]
        return cmd + (["--no-amp"] if NOAMP else [])
    return f
TASKS = []
for model, off_seeds, clean_seeds in JOBS:                      # official runs first (longest), clean runs fill the other GPU
    TASKS += [(f"official_s{s}_{model[:-3]}", make_cmd(model, "official", 0, s), None) for s in off_seeds]
    TASKS += [(f"clean_f{f}_s{s}_{model[:-3]}", make_cmd(model, "clean", f, s), None) for s in clean_seeds for f in FOLDS]
print(len(TASKS), "runs, epochs =", EPOCHS, "batch =", BATCH, "amp =", not NOAMP, "->", OUT)

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count()))
q = queue.Queue()
for t in TASKS:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            name, cmd_fn, needs = q.get_nowait()
        except queue.Empty:
            return
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd_fn(gpu), stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))

In [ ]:
import glob, json, shutil
print(f"{'run':44s} {'mAP50-95':>9s} {'mAP50':>7s} {'epochs':>6s} {'train_min':>9s}  flag")
for p in sorted(glob.glob(f"{OUT}/*/metrics.json")):
    r = json.load(open(p))
    m, flag = r["test"]["map5095"], ""
    if m < 0.05:
        flag = "<-- collapsed? check the log for nan, then rerun with NOAMP = True"
    print(f"{r['tag']:44s} {m:9.4f} {r['test']['map50']:7.4f} {r.get('epochs_run','?'):>6} {r['train_seconds']/60:9.1f}  {flag}")
if SMOKE:
    print("\nSMOKE done. Check: no failed runs, metrics exist. Epoch time ~ train_min / 2 (includes start-up); set SMOKE = False for the real run.")
else:
    subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_benchmark.py", "--runs", OUT, f"{REPO}/results/controls",
                    f"{REPO}/results/baseline_fixed", f"{REPO}/results/bench_A", f"{REPO}/results/bench_B", "--out", f"{OUT}/benchmark_rtdetr_part{PART}"])
shutil.make_archive(f"/kaggle/working/rtdetr_{PART}{'_smoke' if SMOKE else ''}_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if failed:
    print("\nWARNING: failed runs:", failed)

After the real run: download `rtdetr_<PART>_results.zip`, unzip once into `results/rtdetr_<PART>/`, commit and push (the `*.pt` weights are not kept; `*.zip` is git-ignored).
Reading the result: compare `rtdetr-l.pt` clean mAP50-95 with YOLO11s 0.2225, YOLOv8m 0.2271, YOLOv8s 0.2170 (one seed here, so differences below about 1.6 points are not resolvable).